In [1]:
import json
import hashlib
import unicodedata
from pathlib import Path

import pandas as pd

In [2]:
# Google Colab
PATH = Path("/content/data")

# Local — uncomment and adjust for your working directory
# PATH = Path("diplomacy-deception-detection/data/raw")
# PATH = Path("diplomacy-deception-detection/data")  # original Phase 1 ZIP layout

OUTPUT_PATH = PATH.parent / "processed_notebook"

SPLITS = ("train", "validation", "test")

for split in SPLITS:
    filepath = PATH / f"{split}.jsonl"
    print(filepath, "exists:", filepath.is_file())

print("Output folder:", OUTPUT_PATH)

/content/data/train.jsonl exists: True
/content/data/validation.jsonl exists: True
/content/data/test.jsonl exists: True
Output folder: /content/processed_notebook


In [3]:
def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                raise ValueError(f"Blank line in {path}, line {line_number}")
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {path}, line {line_number}") from error
            if not isinstance(record, dict):
                raise ValueError(f"Expected a JSON object in {path}, line {line_number}")
            records.append(record)
    return records

splits = {}
for split in SPLITS:
    splits[split] = read_jsonl(PATH / f"{split}.jsonl")
    print(split, ":", len(splits[split]), "conversation records")

train : 189 conversation records
validation : 21 conversation records
test : 42 conversation records


In [4]:
MESSAGE_FIELDS = (
    "messages", "sender_labels", "receiver_labels", "speakers", "receivers",
    "absolute_message_index", "relative_message_index", "seasons", "years",
    "game_score", "game_score_delta"
)
REQUIRED_FIELDS = MESSAGE_FIELDS + ("game_id", "players")

for split, records in splits.items():
    for conversation_index, record in enumerate(records):
        missing = [field for field in REQUIRED_FIELDS if field not in record]
        if missing:
            raise ValueError(f"{split} conversation {conversation_index}: missing {missing}")
        if any(not isinstance(record[field], list) for field in MESSAGE_FIELDS):
            raise ValueError(f"{split} conversation {conversation_index}: expected message arrays")
        lengths = {field: len(record[field]) for field in MESSAGE_FIELDS}
        if len(set(lengths.values())) != 1:
            raise ValueError(f"{split} conversation {conversation_index}: unequal lengths {lengths}")

print("All records have required fields and aligned message arrays.")

All records have required fields and aligned message arrays.


#### Validate labels and split integrity

In [5]:
game_ids = {}
for split, records in splits.items():
    game_ids[split] = set()
    for conversation_index, record in enumerate(records):
        game_id = record["game_id"]
        if type(game_id) is not int:
            raise ValueError(f"{split} conversation {conversation_index}: invalid game_id")
        game_ids[split].add(game_id)
        if not isinstance(record["players"], list) or len(record["players"]) != 2:
            raise ValueError(f"{split} conversation {conversation_index}: expected two players")
        for i, sender_label in enumerate(record["sender_labels"]):
            if type(sender_label) is not bool:
                raise ValueError(f"Invalid sender label: {split}, {conversation_index}, {i}")
        for i, receiver_label in enumerate(record["receiver_labels"]):
            if type(receiver_label) is not bool and receiver_label != "NOANNOTATION":
                raise ValueError(f"Invalid receiver label: {split}, {conversation_index}, {i}")

for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = game_ids[a] & game_ids[b]
    print(f"{a} / {b} shared games:", len(overlap))
    if overlap:
        raise ValueError(f"Game-level leakage between {a} and {b}: {overlap}")

print("Labels and game splits passed critical checks.")

train / validation shared games: 0
train / test shared games: 0
validation / test shared games: 0
Labels and game splits passed critical checks.


In [6]:
def normalize_text(value):
    return unicodedata.normalize(
        "NFC", value.replace("\r\n", "\n").replace("\r", "\n")
    )

sample_text = "Hello\r\nFrance!"
print("Original:", repr(sample_text))
print("Normalized:", repr(normalize_text(sample_text)))

Original: 'Hello\r\nFrance!'
Normalized: 'Hello\nFrance!'


#### Parse conversation into message records


In [7]:
def parse_conversation(record, split, conversation_index):
    parsed = []
    for i, source in enumerate(record["messages"]):
        parsed.append({
            "split": split,
            "game_id": record["game_id"],
            "conversation_index": conversation_index,
            "message_id": f"{record['game_id']}:{conversation_index}:{i}",
            "message_position": i,
            "message": source,
            "normalized_message": normalize_text(source) if isinstance(source, str) else None,
            "speaker": record["speakers"][i],
            "receiver": record["receivers"][i],
            "sender_label": record["sender_labels"][i],
            "receiver_label": record["receiver_labels"][i],
            "game_score": record["game_score"][i],
            "score_delta": record["game_score_delta"][i],
            "season": record["seasons"][i],
            "year": record["years"][i],
            "absolute_message_index": record["absolute_message_index"][i],
            "relative_message_index": record["relative_message_index"][i],
            "players": record["players"],
        })
    return parsed

example_rows = parse_conversation(splits["train"][0], "train", 0)
print("Messages in first conversation:", len(example_rows))

Messages in first conversation: 321


In [8]:
display(pd.DataFrame(example_rows).head(3))


,split,game_id,conversation_index,message_id,message_position,message,normalized_message,speaker,receiver,sender_label,receiver_label,game_score,score_delta,season,year,absolute_message_index,relative_message_index,players
0,train,1,0,1:0:0,0,Germany!\n\nJust the person I want to speak wi...,Germany!\n\nJust the person I want to speak wi...,italy,germany,True,True,3,0,Spring,1901,74,0,"[italy, germany]"
1,train,1,0,1:0:1,1,"You've whet my appetite, Italy. What's the sug...","You've whet my appetite, Italy. What's the sug...",germany,italy,True,True,3,0,Spring,1901,76,1,"[italy, germany]"
2,train,1,0,1:0:2,2,👍,👍,italy,germany,True,True,3,0,Spring,1901,86,2,"[italy, germany]"


#### Build all message-level datasets

In [9]:
def build_messages(conversations, split):
    rows = []
    for conversation_index, record in enumerate(conversations):
        rows.extend(parse_conversation(record, split, conversation_index))
    return rows

messages_by_split = {}
for split in SPLITS:
    messages_by_split[split] = build_messages(splits[split], split)
    print(split, ":", len(messages_by_split[split]), "messages")

train : 13132 messages
validation : 1416 messages
test : 2741 messages


#### game timeline

In [11]:
def build_game_timelines(messages):
    timelines = {}
    for message in messages:
        timelines.setdefault(message["game_id"], []).append(message)
    for timeline in timelines.values():
        timeline.sort(key=lambda row: (
            int(row["absolute_message_index"]),
            row["conversation_index"], row["message_position"]
        ))
    return timelines

train_timelines = build_game_timelines(messages_by_split["train"])
first_game_id = next(iter(train_timelines))
print("Example game:", first_game_id)

Example game: 1


In [12]:
pd.DataFrame(train_timelines[first_game_id])[ [
    "message_id", "absolute_message_index", "speaker", "receiver", "message"
] ].head(5)

,message_id,absolute_message_index,speaker,receiver,message
0,1:13:0,0,austria,turkey,Hello there! What's your general plan for this...
1,1:1:0,1,austria,italy,Hello there! What's your general plan for this...
2,1:11:0,2,austria,russia,Hello there! What's your general plan for this...
3,1:6:0,3,austria,germany,Hello there! What's your general plan for this...
4,1:14:0,4,austria,france,Hi France! I don't think we have a ton to talk...


#### Create two labeled task datasets

In [13]:
def make_task_dataset(messages, task):
    target_field = {
        "actual_lie": "sender_label",
        "suspected_lie": "receiver_label",
    }.get(task)
    if target_field is None:
        raise ValueError(f"Unknown task: {task}")

    selected = []
    for row in messages:
        if type(row[target_field]) is bool:
            selected.append({
                **row,
                "target": int(row[target_field] is False),
                "target_name": task,
            })
    return selected


In [14]:
tasks = ("actual_lie", "suspected_lie")
processed = {}
for task in tasks:
    processed[task] = {}
    for split in SPLITS:
        selected = make_task_dataset(messages_by_split[split], task)
        processed[task][split] = selected
        positives = sum(row["target"] for row in selected)
        print(f"{task:14s} | {split:10s} | rows={len(selected):5d} | deceptive={positives:5d}")

actual_lie     | train      | rows=13132 | deceptive=  591
actual_lie     | validation | rows= 1416 | deceptive=   56
actual_lie     | test       | rows= 2741 | deceptive=  240
suspected_lie  | train      | rows=12025 | deceptive=  566
suspected_lie  | validation | rows= 1283 | deceptive=   47
suspected_lie  | test       | rows= 2475 | deceptive=  165


#### Verify the processed dataset

In [15]:
for task in tasks:
    source_label = "sender_label" if task == "actual_lie" else "receiver_label"
    for split in SPLITS:
        rows = processed[task][split]
        assert all(type(row[source_label]) is bool for row in rows)
        assert all(row["target"] == int(row[source_label] is False) for row in rows)
        assert all(row["target_name"] == task for row in rows)
        ids = [row["message_id"] for row in rows]
        assert len(ids) == len(set(ids)), f"Duplicate IDs in {task}/{split}"
        if task == "actual_lie":
            assert len(rows) == len(messages_by_split[split])

print("Processed data checks passed.")

Processed data checks passed.


#### Save JSONL outputs


In [16]:
def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

for task in tasks:
    for split in SPLITS:
        output_file = OUTPUT_PATH / task / f"{split}.jsonl"
        if output_file.exists():
            raise FileExistsError(
                f"Output already exists: {output_file}. "
                "Choose a new OUTPUT_PATH to avoid overwriting it."
            )

for task in tasks:
    for split in SPLITS:
        output_file = OUTPUT_PATH / task / f"{split}.jsonl"
        write_jsonl(output_file, processed[task][split])
        print("Created:", output_file)

Created: /content/processed_notebook/actual_lie/train.jsonl
Created: /content/processed_notebook/actual_lie/validation.jsonl
Created: /content/processed_notebook/actual_lie/test.jsonl
Created: /content/processed_notebook/suspected_lie/train.jsonl
Created: /content/processed_notebook/suspected_lie/validation.jsonl
Created: /content/processed_notebook/suspected_lie/test.jsonl


In [17]:
for task in tasks:
    for split in SPLITS:
        output_file = OUTPUT_PATH / task / f"{split}.jsonl"
        loaded = read_jsonl(output_file)
        expected = processed[task][split]
        assert loaded == expected, f"Saved data differs: {task}/{split}"
        print(f"Verified {task}/{split}: {len(loaded)} records")

Verified actual_lie/train: 13132 records
Verified actual_lie/validation: 1416 records
Verified actual_lie/test: 2741 records
Verified suspected_lie/train: 12025 records
Verified suspected_lie/validation: 1283 records
Verified suspected_lie/test: 2475 records


#### Verify  original raw files were not modified

SHA-256 fingerprints

In [18]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

raw_hashes = {split: sha256(PATH / f"{split}.jsonl") for split in SPLITS}
for split, fingerprint in raw_hashes.items():
    print(split, fingerprint)

print("Raw files were opened for reading only; notebook output is under:", OUTPUT_PATH)

train 833d8ef26561bbd1921e9f5e730b2ebea0a59cec24f7082bfdcacb8b6cf30792
validation 6b2ca19991c7596693ea61b34b1be2d6ada3fcf6710472edd4afed841556e571
test 2061ee7f040d6d8016760cb146d7821fef57cfbb4ced868f233e63854257c050
Raw files were opened for reading only; notebook output is under: /content/processed_notebook
